# 02 — Ecommify: Carga Dataset Olist → PostgreSQL + MongoDB Atlas

**Prerrequisito**: Notebook 01 ejecutado exitosamente (esquema DDL creado).

Este notebook:
1. Descarga el dataset Olist desde Kaggle
2. Carga los 8 CSVs en PostgreSQL (Supabase)
3. Carga las colecciones en MongoDB Atlas
4. Valida conteos finales

## 1. Instalar dependencias

In [1]:
!pip install --quiet psycopg2-binary 'pymongo[srv]' kaggle pandas tqdm

## 2. Descargar dataset Olist desde Drive

In [2]:
# ── Montar Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# INSTRUCCIÓN: Sube los 9 CSV de Kaggle a la ruta indicada en DATA_DIR
# Ruta sugerida en tu Drive:  Mi unidad / BigData / brazilian_ecommerce /
DATA = '/content/drive/MyDrive/Maestría Arch/optimizacion dbs/datos'
print(f'Ruta de datos: {DATA}')

Mounted at /content/drive
Ruta de datos: /content/drive/MyDrive/Maestría Arch/optimizacion dbs/datos


## 3. Conectar a ambas bases de datos

In [3]:
import psycopg2, getpass
from psycopg2.extras import execute_batch, Json
from pymongo import MongoClient
import pandas as pd, json as json_lib, uuid
from datetime import datetime, timezone
from google.colab import userdata


SUPABASE_URI = userdata.get('SUPABASE_URI')
#getpass.getpass('Supabase URI: ').strip()
ATLAS_URI    = userdata.get('ATLAS_URI')
#getpass.getpass('Atlas URI: ').strip()

pg = psycopg2.connect(SUPABASE_URI, connect_timeout=15,
                      options='-c statement_timeout=300000')
mc = MongoClient(ATLAS_URI, serverSelectionTimeoutMS=10000)
db = mc['ecommify']

# Quick verification
with pg.cursor() as cur:
    cur.execute('SELECT postgis_version();')
    print(f'PG PostGIS: {cur.fetchone()[0]}')
print(f'MongoDB: {mc.server_info()["version"]}')
print('Ambas conexiones activas')

PG PostGIS: 3.3 USE_GEOS=1 USE_PROJ=1 USE_STATS=1
MongoDB: 8.0.26
Ambas conexiones activas


## 4. Carga geolocation (deduplicado por zip)

In [4]:
# DATA= '/content/data'
BATCH = 1000

print('Cargando geolocation...')
df_geo = pd.read_csv(f'{DATA}/olist_geolocation_dataset.csv')
print(f'  Filas originales: {len(df_geo):,}')

# Deduplicar por zip_code_prefix (mediana de lat/lng)
dedup = df_geo.groupby('geolocation_zip_code_prefix').agg(
    geolocation_lat=('geolocation_lat','median'),
    geolocation_lng=('geolocation_lng','median'),
    geolocation_city=('geolocation_city','first'),
    geolocation_state=('geolocation_state','first')
).reset_index()
print(f'  Filas deduplicadas: {len(dedup):,}')

rows_pg = [
    (int(r.geolocation_zip_code_prefix), float(r.geolocation_lat), float(r.geolocation_lng),
     str(r.geolocation_city)[:60], str(r.geolocation_state)[:2])
    for r in dedup.itertuples()
    if -34 <= r.geolocation_lat <= 5.3 and -74 <= r.geolocation_lng <= -34
]

with pg.cursor() as cur:
    execute_batch(cur, """
        INSERT INTO geolocation (zip_code_prefix, geolocation_lat, geolocation_lng,
                                 geolocation_city, geolocation_state)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (zip_code_prefix) DO NOTHING
    """, rows_pg, page_size=BATCH)
pg.commit()
print(f'  PG geolocation: {len(rows_pg):,} filas insertadas')

# MongoDB geolocation (FULL — con 2dsphere)
docs_geo = []
for r in df_geo.itertuples():
    if -34 <= r.geolocation_lat <= 5.3 and -74 <= r.geolocation_lng <= -34:
        docs_geo.append({
            'zip_code_prefix': int(r.geolocation_zip_code_prefix),
            'city': str(r.geolocation_city)[:60],
            'state': str(r.geolocation_state)[:2],
            'location': {'type': 'Point',
                         'coordinates': [float(r.geolocation_lng), float(r.geolocation_lat)]}
        })

# Insert en batches de 5000
coll_geo = db.geolocation
coll_geo.drop()
coll_geo.create_index([('location', '2dsphere')])
coll_geo.create_index([('zip_code_prefix', 1)], unique=True)

for i in range(0, len(docs_geo), 5000):
    coll_geo.insert_many(docs_geo[i:i+5000], ordered=False)
print(f'  MongoDB geolocation: {coll_geo.count_documents({}):,} documentos')

Cargando geolocation...
  Filas originales: 1,000,163
  Filas deduplicadas: 19,015
  PG geolocation: 19,007 filas insertadas


BulkWriteError: batch op errors occurred, full error: {'writeErrors': [{'index': 2, 'code': 11000, 'errmsg': 'E11000 duplicate key error collection: ecommify.geolocation index: zip_code_prefix_1 dup key: { zip_code_prefix: 1046 }', 'keyPattern': {'zip_code_prefix': 1}, 'keyValue': {'zip_code_prefix': 1046}, 'op': {'zip_code_prefix': 1046, 'city': 'sao paulo', 'state': 'SP', 'location': {'type': 'Point', 'coordinates': [-46.64295148361138, -23.54612896641469]}, '_id': ObjectId('6a2f4814e17db01cbef36c00')}}, {'index': 10, 'code': 11000, 'errmsg': 'E11000 duplicate key error collection: ecommify.geolocation index: zip_code_prefix_1 dup key: { zip_code_prefix: 1013 }', 'keyPattern': {'zip_code_prefix': 1}, 'keyValue': {'zip_code_prefix': 1013}, 'op': {'zip_code_prefix': 1013, 'city': 'sao paulo', 'state': 'SP', 'location': {'type': 'Point', 'coordinates': [-46.63418378613892, -23.547325128224376]}, '_id': ObjectId('6a2f4814e17db01cbef36c08')}}, {'index': 13, 'code': 11000, 'errmsg': 'E1100

## 5. Carga categorías, customers, sellers, products

In [6]:
# ── Categorías ──
print('Cargando categorías...')
df_cat = pd.read_csv(f'{DATA}/product_category_name_translation.csv')
with pg.cursor() as cur:
    execute_batch(cur,
        'INSERT INTO product_category_name_translation VALUES (%s,%s) ON CONFLICT DO NOTHING',
        df_cat.itertuples(index=False), page_size=BATCH)
pg.commit()
print(f'  Categorías: {len(df_cat)} filas')

# ── Customers ──
print('Cargando customers...')
df_cust = pd.read_csv(f'{DATA}/olist_customers_dataset.csv')

# Obtener zips disponibles en PG para respetar FK
with pg.cursor() as cur:
    cur.execute('SELECT zip_code_prefix FROM geolocation')
    valid_zips = {r[0] for r in cur.fetchall()}

rows_cust, cust_id_map = [], {}
for r in df_cust.itertuples():
    if int(r.customer_zip_code_prefix) in valid_zips:
        cuid = str(uuid.uuid4())
        try:
            u_uuid = str(uuid.UUID(r.customer_unique_id))
        except:
            u_uuid = str(uuid.uuid4())
        cust_id_map[r.customer_id] = (cuid, u_uuid)
        rows_cust.append((cuid, u_uuid, int(r.customer_zip_code_prefix),
                          str(r.customer_city)[:60], str(r.customer_state)[:2]))

with pg.cursor() as cur:
    execute_batch(cur, """
        INSERT INTO customers(customer_id, customer_unique_id, zip_code_prefix, customer_city, customer_state)
        VALUES (%s, %s, %s, %s, %s)
    """, rows_cust, page_size=BATCH)
pg.commit()
print(f'  Customers: {len(rows_cust):,} filas (de {len(df_cust):,} originales)')

# ── Sellers (sin FK geo) ──
print('Cargando sellers...')
df_sell = pd.read_csv(f'{DATA}/olist_sellers_dataset.csv')
seller_id_map = {}
rows_sell = []
for r in df_sell.itertuples():
    sid = str(uuid.uuid4())
    seller_id_map[r.seller_id] = sid
    rows_sell.append((sid, int(r.seller_zip_code_prefix),
                      str(r.seller_city)[:60], str(r.seller_state)[:2]))
with pg.cursor() as cur:
    execute_batch(cur,
        'INSERT INTO sellers(seller_id, seller_zip_code_prefix, seller_city, seller_state) VALUES (%s,%s,%s,%s)',
        rows_sell, page_size=BATCH)
pg.commit()
print(f'  Sellers: {len(rows_sell):,} filas')

# ── Products ──
print('Cargando products...')
df_prod = pd.read_csv(f'{DATA}/olist_products_dataset.csv')
product_id_map = {}
rows_prod = []
for r in df_prod.itertuples():
    pid = str(uuid.uuid4())
    product_id_map[r.product_id] = pid
    cat = None if pd.isna(r.product_category_name) else str(r.product_category_name)[:100]
    nl  = None if pd.isna(r.product_name_lenght) else int(r.product_name_lenght)
    dl  = None if pd.isna(r.product_description_lenght) else int(r.product_description_lenght)
    pq  = None if pd.isna(r.product_photos_qty) else int(r.product_photos_qty)
    dims = None
    if not pd.isna(r.product_length_cm):
        dims = (float(r.product_length_cm), float(r.product_height_cm),
                float(r.product_width_cm), int(r.product_weight_g))
    rows_prod.append((pid, cat, nl, dl, pq, dims))

with pg.cursor() as cur:
    execute_batch(cur, """
        INSERT INTO products(product_id, product_category_name, product_name_length,
            product_description_length, product_photos_qty, dims)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, rows_prod, page_size=BATCH)
pg.commit()
print(f'  Products: {len(rows_prod):,} filas')

Cargando categorías...


InFailedSqlTransaction: current transaction is aborted, commands ignored until end of transaction block


Cargar productos: por foreingkey, ejecutar rollback en caso de falla para volver a lanzar

In [11]:
# ── Products ──
print('Cargando products...')
df_prod = pd.read_csv(f'{DATA}/olist_products_dataset.csv')

product_id_map = {}
rows_prod = []

# Categorías únicas del archivo products
cats = (
    df_prod['product_category_name']
    .dropna()
    .astype(str)
    .str[:100]
    .drop_duplicates()
)

rows_cat_missing = [(c, c) for c in cats]

try:
    with pg.cursor() as cur:

        # 1. Crear categorías faltantes para evitar error de llave foránea
        execute_batch(cur, """
            INSERT INTO product_category_name_translation
                (product_category_name, product_category_name_english)
            VALUES (%s, %s)
            ON CONFLICT (product_category_name) DO NOTHING
        """, rows_cat_missing, page_size=BATCH)

        # 2. Preparar productos
        for r in df_prod.itertuples(index=False):
            pid = str(uuid.uuid4())
            product_id_map[r.product_id] = pid

            cat = None if pd.isna(r.product_category_name) else str(r.product_category_name)[:100]
            nl  = None if pd.isna(r.product_name_lenght) else int(r.product_name_lenght)
            dl  = None if pd.isna(r.product_description_lenght) else int(r.product_description_lenght)
            pq  = None if pd.isna(r.product_photos_qty) else int(r.product_photos_qty)

            dims = None
            if not pd.isna(r.product_length_cm):
                dims = (
                    float(r.product_length_cm),
                    float(r.product_height_cm),
                    float(r.product_width_cm),
                    int(r.product_weight_g)
                )

            rows_prod.append((pid, cat, nl, dl, pq, dims))

        # 3. Insertar productos
        execute_batch(cur, """
            INSERT INTO products(
                product_id,
                product_category_name,
                product_name_length,
                product_description_length,
                product_photos_qty,
                dims
            )
            VALUES (%s, %s, %s, %s, %s, %s)
        """, rows_prod, page_size=BATCH)

    pg.commit()

except Exception as e:
    pg.rollback()
    print("Error cargando products:", e)
    raise

print(f'  Categorías validadas/creadas: {len(rows_cat_missing):,}')
print(f'  Products: {len(rows_prod):,} filas')

Cargando products...
  Categorías validadas/creadas: 73
  Products: 32,951 filas


In [21]:
pg.rollback()

## 6. Carga orders + items + payments

In [22]:
#pg.rollback()

# Validar que los mapas del bloque anterior existan
required_maps = ['cust_id_map', 'product_id_map', 'seller_id_map']
for m in required_maps:
    if m not in globals():
        raise Exception(f"Falta {m}. Ejecuta primero el bloque de customers, sellers y products.")


# ── orders ──
print('Cargando orders...')

df_ord = pd.read_csv(
    f'{DATA}/olist_orders_dataset.csv',
    parse_dates=[
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
)

valid_order_status = {
    'delivered',
    'shipped',
    'canceled',
    'approved',
    'processing',
    'unavailable',
    'invoiced',
    'created'
}

order_id_map = {}
rows_ord_stage = []

skipped_orders_map = 0
skipped_orders_check_py = 0

for r in df_ord.itertuples(index=False):

    if r.customer_id not in cust_id_map:
        skipped_orders_map += 1
        continue

    order_status = str(r.order_status)[:20]

    # Validación CHECK en Python antes de stage
    if order_status not in valid_order_status:
        skipped_orders_check_py += 1
        continue

    oid = str(uuid.uuid4())
    customer_uuid = cust_id_map[r.customer_id][0]

    order_id_map[r.order_id] = (oid, r.order_purchase_timestamp)

    rows_ord_stage.append((
        oid,
        customer_uuid,
        order_status,
        r.order_purchase_timestamp,
        None if pd.isna(r.order_approved_at) else r.order_approved_at,
        None if pd.isna(r.order_delivered_carrier_date) else r.order_delivered_carrier_date,
        None if pd.isna(r.order_delivered_customer_date) else r.order_delivered_customer_date,
        None if pd.isna(r.order_estimated_delivery_date) else r.order_estimated_delivery_date
    ))

try:
    with pg.cursor() as cur:

        cur.execute("""
            CREATE TEMP TABLE tmp_orders_stage (
                order_id UUID,
                customer_id UUID,
                order_status VARCHAR(20),
                order_purchase_timestamp TIMESTAMPTZ,
                order_approved_at TIMESTAMPTZ,
                order_delivered_carrier_date TIMESTAMPTZ,
                order_delivered_customer_date TIMESTAMPTZ,
                order_estimated_delivery_date TIMESTAMPTZ
            ) ON COMMIT DROP
        """)

        execute_batch(cur, """
            INSERT INTO tmp_orders_stage
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
        """, rows_ord_stage, page_size=BATCH)

        cur.execute("""
            INSERT INTO orders(
                order_id,
                customer_id,
                order_status,
                order_purchase_timestamp,
                order_approved_at,
                order_delivered_carrier_date,
                order_delivered_customer_date,
                order_estimated_delivery_date
            )
            SELECT
                t.order_id,
                t.customer_id,
                t.order_status,
                t.order_purchase_timestamp,
                t.order_approved_at,
                t.order_delivered_carrier_date,
                t.order_delivered_customer_date,
                t.order_estimated_delivery_date
            FROM tmp_orders_stage t
            JOIN customers c
              ON c.customer_id = t.customer_id
            WHERE t.order_status IN (
                'delivered',
                'shipped',
                'canceled',
                'approved',
                'processing',
                'unavailable',
                'invoiced',
                'created'
            )
            ON CONFLICT DO NOTHING
        """)

        inserted_orders = cur.rowcount

        cur.execute("""
            SELECT
                COUNT(*) FILTER (WHERE c.customer_id IS NULL) AS sin_customer_fk,
                COUNT(*) FILTER (
                    WHERE t.order_status NOT IN (
                        'delivered',
                        'shipped',
                        'canceled',
                        'approved',
                        'processing',
                        'unavailable',
                        'invoiced',
                        'created'
                    )
                ) AS check_status_invalido
            FROM tmp_orders_stage t
            LEFT JOIN customers c
              ON c.customer_id = t.customer_id
        """)

        skipped_orders_fk, skipped_orders_check_sql = cur.fetchone()

    pg.commit()

except Exception as e:
    pg.rollback()
    print("Error cargando orders:", e)
    raise

print(f'  Orders preparadas: {len(rows_ord_stage):,}')
print(f'  Orders insertadas: {inserted_orders:,}')
print(f'  Orders saltadas sin mapa customer: {skipped_orders_map:,}')
print(f'  Orders saltadas por FK customer: {skipped_orders_fk:,}')
print(f'  Orders saltadas por CHECK status en Python: {skipped_orders_check_py:,}')
print(f'  Orders saltadas por CHECK status en SQL: {skipped_orders_check_sql:,}')


# ── order_items ──
print('Cargando order_items...')

df_items = pd.read_csv(
    f'{DATA}/olist_order_items_dataset.csv',
    parse_dates=['shipping_limit_date']
)

rows_items_stage = []

skipped_items_order_map = 0
skipped_items_product_map = 0
skipped_items_seller_map = 0
skipped_items_check_py = 0

for r in df_items.itertuples(index=False):

    if r.order_id not in order_id_map:
        skipped_items_order_map += 1
        continue

    if r.product_id not in product_id_map:
        skipped_items_product_map += 1
        continue

    if r.seller_id not in seller_id_map:
        skipped_items_seller_map += 1
        continue

    order_item_id = int(r.order_item_id)
    price = float(r.price)
    freight_value = float(r.freight_value)

    # Validación CHECK en Python antes de stage
    if order_item_id <= 0 or price < 0 or freight_value < 0:
        skipped_items_check_py += 1
        continue

    oid, ots = order_id_map[r.order_id]

    rows_items_stage.append((
        oid,
        ots,
        order_item_id,
        product_id_map[r.product_id],
        seller_id_map[r.seller_id],
        r.shipping_limit_date,
        price,
        freight_value
    ))

try:
    with pg.cursor() as cur:

        cur.execute("""
            CREATE TEMP TABLE tmp_order_items_stage (
                order_id UUID,
                order_purchase_timestamp TIMESTAMPTZ,
                order_item_id SMALLINT,
                product_id UUID,
                seller_id UUID,
                shipping_limit_date TIMESTAMPTZ,
                price NUMERIC(10,2),
                freight_value NUMERIC(10,2)
            ) ON COMMIT DROP
        """)

        execute_batch(cur, """
            INSERT INTO tmp_order_items_stage
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
        """, rows_items_stage, page_size=BATCH)

        cur.execute("""
            INSERT INTO order_items(
                order_id,
                order_purchase_timestamp,
                order_item_id,
                product_id,
                seller_id,
                shipping_limit_date,
                price,
                freight_value
            )
            SELECT
                t.order_id,
                t.order_purchase_timestamp,
                t.order_item_id,
                t.product_id,
                t.seller_id,
                t.shipping_limit_date,
                t.price,
                t.freight_value
            FROM tmp_order_items_stage t
            JOIN orders o
              ON o.order_id = t.order_id
             AND o.order_purchase_timestamp = t.order_purchase_timestamp
            JOIN products p
              ON p.product_id = t.product_id
            JOIN sellers s
              ON s.seller_id = t.seller_id
            WHERE t.order_item_id > 0
              AND t.price >= 0
              AND t.freight_value >= 0
            ON CONFLICT DO NOTHING
        """)

        inserted_items = cur.rowcount

        cur.execute("""
            SELECT
                COUNT(*) FILTER (WHERE o.order_id IS NULL) AS sin_order,
                COUNT(*) FILTER (WHERE p.product_id IS NULL) AS sin_product,
                COUNT(*) FILTER (WHERE s.seller_id IS NULL) AS sin_seller,
                COUNT(*) FILTER (
                    WHERE t.order_item_id <= 0
                       OR t.price < 0
                       OR t.freight_value < 0
                ) AS check_invalido
            FROM tmp_order_items_stage t
            LEFT JOIN orders o
              ON o.order_id = t.order_id
             AND o.order_purchase_timestamp = t.order_purchase_timestamp
            LEFT JOIN products p
              ON p.product_id = t.product_id
            LEFT JOIN sellers s
              ON s.seller_id = t.seller_id
        """)

        (
            skipped_items_fk_order,
            skipped_items_fk_product,
            skipped_items_fk_seller,
            skipped_items_check_sql
        ) = cur.fetchone()

    pg.commit()

except Exception as e:
    pg.rollback()
    print("Error cargando order_items:", e)
    raise

print(f'  Order_items preparadas: {len(rows_items_stage):,}')
print(f'  Order_items insertadas: {inserted_items:,}')
print(f'  Order_items saltadas sin mapa order: {skipped_items_order_map:,}')
print(f'  Order_items saltadas sin mapa product: {skipped_items_product_map:,}')
print(f'  Order_items saltadas sin mapa seller: {skipped_items_seller_map:,}')
print(f'  Order_items saltadas por FK order: {skipped_items_fk_order:,}')
print(f'  Order_items saltadas por FK product: {skipped_items_fk_product:,}')
print(f'  Order_items saltadas por FK seller: {skipped_items_fk_seller:,}')
print(f'  Order_items saltadas por CHECK en Python: {skipped_items_check_py:,}')
print(f'  Order_items saltadas por CHECK en SQL: {skipped_items_check_sql:,}')


# ── order_payments ──
print('Cargando order_payments...')

df_pay = pd.read_csv(f'{DATA}/olist_order_payments_dataset.csv')

valid_payment_type = {
    'credit_card',
    'debit_card',
    'boleto',
    'voucher',
    'not_defined'
}

rows_pay_stage = []

skipped_pay_order_map = 0
skipped_pay_check_py = 0

for r in df_pay.itertuples(index=False):

    if r.order_id not in order_id_map:
        skipped_pay_order_map += 1
        continue

    payment_sequential = int(r.payment_sequential)
    payment_type = str(r.payment_type)[:15]
    payment_installments = int(r.payment_installments)
    payment_value = float(r.payment_value)

    # Validación CHECK en Python antes de stage
    if (
        payment_sequential < 1
        or payment_type not in valid_payment_type
        or payment_installments < 1
        or payment_value <= 0
    ):
        skipped_pay_check_py += 1
        continue

    oid, ots = order_id_map[r.order_id]

    rows_pay_stage.append((
        oid,
        ots,
        payment_sequential,
        payment_type,
        payment_installments,
        payment_value
    ))

try:
    with pg.cursor() as cur:

        cur.execute("""
            CREATE TEMP TABLE tmp_order_payments_stage (
                order_id UUID,
                order_purchase_timestamp TIMESTAMPTZ,
                payment_sequential SMALLINT,
                payment_type VARCHAR(15),
                payment_installments SMALLINT,
                payment_value NUMERIC(10,2)
            ) ON COMMIT DROP
        """)

        execute_batch(cur, """
            INSERT INTO tmp_order_payments_stage
            VALUES (%s,%s,%s,%s,%s,%s)
        """, rows_pay_stage, page_size=BATCH)

        cur.execute("""
            INSERT INTO order_payments(
                order_id,
                order_purchase_timestamp,
                payment_sequential,
                payment_type,
                payment_installments,
                payment_value
            )
            SELECT
                t.order_id,
                t.order_purchase_timestamp,
                t.payment_sequential,
                t.payment_type,
                t.payment_installments,
                t.payment_value
            FROM tmp_order_payments_stage t
            JOIN orders o
              ON o.order_id = t.order_id
             AND o.order_purchase_timestamp = t.order_purchase_timestamp
            WHERE t.payment_sequential >= 1
              AND t.payment_type IN (
                    'credit_card',
                    'debit_card',
                    'boleto',
                    'voucher',
                    'not_defined'
              )
              AND t.payment_installments >= 1
              AND t.payment_value > 0
            ON CONFLICT DO NOTHING
        """)

        inserted_payments = cur.rowcount

        cur.execute("""
            SELECT
                COUNT(*) FILTER (WHERE o.order_id IS NULL) AS sin_order,
                COUNT(*) FILTER (
                    WHERE t.payment_sequential < 1
                       OR t.payment_type NOT IN (
                            'credit_card',
                            'debit_card',
                            'boleto',
                            'voucher',
                            'not_defined'
                       )
                       OR t.payment_installments < 1
                       OR t.payment_value <= 0
                ) AS check_invalido,
                COUNT(*) FILTER (WHERE t.payment_value <= 0) AS payment_value_invalido
            FROM tmp_order_payments_stage t
            LEFT JOIN orders o
              ON o.order_id = t.order_id
             AND o.order_purchase_timestamp = t.order_purchase_timestamp
        """)

        (
            skipped_payments_fk_order,
            skipped_payments_check_sql,
            skipped_payments_value_sql
        ) = cur.fetchone()

    pg.commit()

except Exception as e:
    pg.rollback()
    print("Error cargando order_payments:", e)
    raise

print(f'  Order_payments preparadas: {len(rows_pay_stage):,}')
print(f'  Order_payments insertadas: {inserted_payments:,}')
print(f'  Order_payments saltadas sin mapa order: {skipped_pay_order_map:,}')
print(f'  Order_payments saltadas por FK order: {skipped_payments_fk_order:,}')
print(f'  Order_payments saltadas por CHECK en Python: {skipped_pay_check_py:,}')
print(f'  Order_payments saltadas por CHECK en SQL: {skipped_payments_check_sql:,}')
print(f'  Order_payments saltadas por payment_value <= 0: {skipped_payments_value_sql:,}')

Cargando orders...
  Orders preparadas: 99,159
  Orders insertadas: 95,824
  Orders saltadas sin mapa customer: 282
  Orders saltadas por FK customer: 3,335
  Orders saltadas por CHECK status en Python: 0
  Orders saltadas por CHECK status en SQL: 0
Cargando order_items...
  Order_items preparadas: 112,344
  Order_items insertadas: 108,307
  Order_items saltadas sin mapa order: 306
  Order_items saltadas sin mapa product: 0
  Order_items saltadas sin mapa seller: 0
  Order_items saltadas por FK order: 4,037
  Order_items saltadas por FK product: 0
  Order_items saltadas por FK seller: 0
  Order_items saltadas por CHECK en Python: 0
  Order_items saltadas por CHECK en SQL: 0
Cargando order_payments...
  Order_payments preparadas: 103,583
  Order_payments insertadas: 100,064
  Order_payments saltadas sin mapa order: 292
  Order_payments saltadas por FK order: 3,519
  Order_payments saltadas por CHECK en Python: 11
  Order_payments saltadas por CHECK en SQL: 0
  Order_payments saltadas po

## 7. Cargar reviews en MongoDB

In [23]:
print('Cargando reviews en MongoDB...')
df_rev = pd.read_csv(f'{DATA}/olist_order_reviews_dataset.csv',
    parse_dates=['review_creation_date','review_answer_timestamp'])

docs_rev = []
for r in df_rev.itertuples():
    docs_rev.append({
        'review_id': str(r.review_id),
        'order_id':  str(r.order_id),
        'review_score': int(r.review_score),
        'review_comment_title':   None if pd.isna(r.review_comment_title) else str(r.review_comment_title),
        'review_comment_message': None if pd.isna(r.review_comment_message) else str(r.review_comment_message),
        'review_creation_date':   r.review_creation_date.to_pydatetime() if not pd.isna(r.review_creation_date) else None,
        'review_answer_timestamp':r.review_answer_timestamp.to_pydatetime() if not pd.isna(r.review_answer_timestamp) else None,
    })

coll_rev = db.reviews
coll_rev.drop()
# Crear índices ANTES de cargar (evita crear en colección llena)
coll_rev.create_index([('review_id', 1)], unique=True)
coll_rev.create_index([('order_id', 1)])
coll_rev.create_index([('review_score', 1)])
coll_rev.create_index([('review_comment_message', 'text'),
                        ('review_comment_title', 'text')],
                       default_language='portuguese')

for i in range(0, len(docs_rev), 5000):
    try:
        coll_rev.insert_many(docs_rev[i:i+5000], ordered=False)
    except Exception as e:
        pass  # duplicados ignorados
print(f'  MongoDB reviews: {coll_rev.count_documents({}):,} documentos')
pct_null = coll_rev.count_documents({'review_comment_message': None}) / coll_rev.count_documents({}) * 100
print(f'  Sin comentario: {pct_null:.1f}% (EDA esperado: ~41.1%)')

Cargando reviews en MongoDB...
  MongoDB reviews: 98,410 documentos
  Sin comentario: 58.7% (EDA esperado: ~41.1%)


## 8. ETL PostgreSQL → MongoDB orders_summary

In [24]:
from pymongo import UpdateOne
print('Ejecutando ETL PG → MongoDB orders_summary...')

SQL_SUMMARY = """
SELECT o.order_id::TEXT, o.order_status, o.order_purchase_timestamp,
       c.customer_id::TEXT, c.customer_state, c.customer_city,
       COALESCE(
           json_agg(
               json_build_object('product_id', oi.product_id::TEXT,
                                 'seller_id', oi.seller_id::TEXT,
                                 'price', oi.price::FLOAT,
                                 'freight_value', oi.freight_value::FLOAT)
               ORDER BY oi.order_item_id
           ) FILTER (WHERE oi.order_id IS NOT NULL),
           '[]'::json
       ) AS items,
       COALESCE(SUM(op.payment_value), 0)::FLOAT AS payment_total,
       (ARRAY_AGG(op.payment_type ORDER BY op.payment_value DESC) FILTER (WHERE op.payment_type IS NOT NULL))[1] AS payment_type_main
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
LEFT JOIN order_items oi ON oi.order_id = o.order_id AND oi.order_purchase_timestamp = o.order_purchase_timestamp
LEFT JOIN order_payments op ON op.order_id = o.order_id AND op.order_purchase_timestamp = o.order_purchase_timestamp
GROUP BY o.order_id, o.order_status, o.order_purchase_timestamp,
         c.customer_id, c.customer_state, c.customer_city
"""

from psycopg2.extras import RealDictCursor
pg2 = psycopg2.connect(SUPABASE_URI, cursor_factory=RealDictCursor,
                       options='-c statement_timeout=600000')

with pg2.cursor() as cur:
    cur.execute(SQL_SUMMARY)
    rows = cur.fetchall()
print(f'  Filas extraídas de PG: {len(rows):,}')

coll_sum = db.orders_summary
coll_sum.drop()
coll_sum.create_index([('status', 1)])
coll_sum.create_index([('purchase_date', -1)])
coll_sum.create_index([('customer.state', 1)])
coll_sum.create_index([('etl_updated_at', 1)])

now = datetime.now(timezone.utc)
ops, batch_size = [], 500
for r in rows:
    doc = {
        '_id': r['order_id'],
        'status': r['order_status'],
        'purchase_date': r['order_purchase_timestamp'],
        'customer': {'customer_id': r['customer_id'],
                     'state': r['customer_state'], 'city': r['customer_city']},
        'items': r['items'] if r['items'] else [],
        'payment_total': r['payment_total'] or 0.0,
        'payment_type_main': r['payment_type_main'],
        'review_score': None,
        'etl_updated_at': now,
    }
    ops.append(UpdateOne({'_id': doc['_id']}, {'$set': doc}, upsert=True))
    if len(ops) >= batch_size:
        coll_sum.bulk_write(ops, ordered=False); ops = []
if ops:
    coll_sum.bulk_write(ops, ordered=False)

print(f'  MongoDB orders_summary: {coll_sum.count_documents({}):,} documentos')
pg2.close()

Ejecutando ETL PG → MongoDB orders_summary...
  Filas extraídas de PG: 191,648
  MongoDB orders_summary: 191,648 documentos


## 9. Validación final de conteos

In [25]:
print('=== RESUMEN FINAL DE CARGA ===')
print('\n--- PostgreSQL (Supabase) ---')
with pg.cursor() as cur:
    for t in ['geolocation','product_category_name_translation','customers',
              'sellers','products','orders','order_items','order_payments']:
        cur.execute(f'SELECT COUNT(*) FROM {t}')
        n = cur.fetchone()[0]
        expected = {'geolocation':19015,'customers':99441,'sellers':3095,
                    'products':32951,'orders':99441,'order_items':112650,
                    'order_payments':103886,'product_category_name_translation':71}
        status = 'OK' if n > 0 else 'VACIA'
        print(f'  {t:40s}: {n:>10,}  [{status}]')

print('\n--- MongoDB Atlas ---')
for col in ['geolocation','reviews','orders_summary']:
    n = db[col].count_documents({})
    print(f'  {col:40s}: {n:>10,}')

pg.close()
mc.close()
print('\nNotebook 02 completado. Ir a notebook 03 para benchmarks.')

=== RESUMEN FINAL DE CARGA ===

--- PostgreSQL (Supabase) ---
  geolocation                             :     19,007  [OK]
  product_category_name_translation       :         73  [OK]
  customers                               :     95,824  [OK]
  sellers                                 :      3,095  [OK]
  products                                :     32,951  [OK]
  orders                                  :    191,648  [OK]
  order_items                             :    216,614  [OK]
  order_payments                          :    100,064  [OK]

--- MongoDB Atlas ---
  geolocation                             :        152
  reviews                                 :     98,410
  orders_summary                          :    191,648

Notebook 02 completado. Ir a notebook 03 para benchmarks.
